# MOCID on Colab — reproduction driver

Runs the Phase 1 baseline from [PLAN.md](../PLAN.md): environment setup, param
check, smoke test, then the R0 two-stage training run on DAUB.

**Prerequisites on Google Drive** (`DRIVE_ROOT` below):
- `data/DAUB_mocid/` in MOCID layout: `train_DAUB.txt`, `val_DAUB.txt`, `images/{train,test}/dataN/*.bmp`
  (build it once with the helper scripts — see the *First-time DAUB prep* cell).
- `runs/` is created automatically; checkpoints land here so a session timeout is recoverable.

**After a timeout:** re-run the *mount*, *config*, *repo*, *deps*, *env* cells, then the *train* cell — it auto-resumes.


## 1. Runtime check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda,
      '| gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Config — edit these

In [ ]:
import os

REPO_URL   = ''                                   # e.g. https://github.com/<you>/IRSTD-Methods.git ; '' -> copy from Drive
REPO_DIR   = '/content/IRSTD-Methods'
DRIVE_ROOT = '/content/drive/MyDrive/MOCID'       # persistent: data/, runs/
DATASET    = 'DAUB'                               # DAUB | IRDST
RUN_TAG    = 'r0-baseline'

DATA_ROOT  = '/content/data'                      # local fast disk (staged from Drive)
RUNS_DIR   = DRIVE_ROOT + '/runs'
VMAMBA_DIR = '/content/VMamba'
MOCID_DIR  = REPO_DIR + '/MOCID'
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)


## 4. Get the repo

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    else:
        src = DRIVE_ROOT + '/IRSTD-Methods-master'
        assert os.path.isdir(src), f'no repo at {src} and REPO_URL is empty'
        subprocess.run(['cp', '-r', src, REPO_DIR], check=True)
%cd {MOCID_DIR}
!git log --oneline -1


## 5. Dependencies + VMamba selective-scan kernel

In [ ]:
!pip -q install opencv-python-headless einops timm

import os
if not os.path.isdir(VMAMBA_DIR):
    # pin a commit here if the csms6s import ever breaks
    !git clone --depth 1 https://github.com/MzeroMiko/VMamba.git {VMAMBA_DIR}

# optional CUDA kernel; csms6s falls back to a pure-pytorch scan if this fails
%cd {VMAMBA_DIR}/kernels/selective_scan
!pip -q install . || echo '[warn] kernel build failed - using the pytorch fallback (slower, same math)'
%cd {MOCID_DIR}


## 6. Environment + point `runs/` at Drive

In [ ]:
import os
os.environ['VMAMBA_PATH']     = VMAMBA_DIR
os.environ['MOCID_DATASET']   = DATASET
os.environ['MOCID_DATA_ROOT'] = DATA_ROOT

!rm -rf {MOCID_DIR}/runs
!ln -s {RUNS_DIR} {MOCID_DIR}/runs
!ls -la {MOCID_DIR}/runs


## First-time DAUB prep (run once, then keep `data/DAUB_mocid/` on Drive)

Needs the raw DAUB `dataN/` folders and the coco split lists shipped in the repo
(`dataset_helpers/coco_train_DAUB.txt`, `coco_val_DAUB.txt`).


In [ ]:
# %cd {MOCID_DIR}/dataset_helpers
# !python reorganize_daub.py \
#     --raw   {DRIVE_ROOT}/raw/DAUB \
#     --train-ann coco_train_DAUB.txt \
#     --val-ann   coco_val_DAUB.txt \
#     --out   {DRIVE_ROOT}/data/DAUB_mocid \
#     --mode  copy
# # -> writes DAUB_mocid/{train_DAUB.txt,val_DAUB.txt} + images/. Paths in the txt are absolute.
# %cd {MOCID_DIR}


## 7. Stage the dataset to local disk

In [ ]:
!rsync -a --info=progress2 {DRIVE_ROOT}/data/{DATASET}_mocid {DATA_ROOT}/
!wc -l {DATA_ROOT}/{DATASET}_mocid/*_{DATASET}.txt
!head -n2 {DATA_ROOT}/{DATASET}_mocid/train_{DATASET}.txt
# NOTE: if the txt paths are absolute and point at the Drive location, either keep the
# data on Drive (slower) or sed-rewrite the prefix to {DATA_ROOT} after rsync.


## 8. Phase 1.1 — parameter count (free gate)
Must be ~9.45 M (no DAM) and ~13.05 M (MOCID). If not, **stop** — architecture is wrong.


In [ ]:
%cd {MOCID_DIR}
!python main.py params


## 9. Phase 1.3 — smoke test (no data)
Validates the VMamba import, all shapes, and a finite fwd/bwd for both branches.


In [ ]:
import torch
from model import MOCID

m = MOCID(num_classes=1, num_frames=5, img_size=512).cuda()
clip = torch.randn(1, 5, 3, 512, 512, device='cuda')
labels = [torch.tensor([[256., 256., 8., 8., 0.]], device='cuda')]  # cx,cy,w,h,cls

for use_dam in (False, True):
    m.train(); m.zero_grad(set_to_none=True)
    loss = m(clip, labels, use_dam=use_dam)
    loss.backward()
    print(f'use_dam={use_dam}  loss={loss.item():.4f}  finite={bool(torch.isfinite(loss))}')

m.eval()
with torch.no_grad():
    outs = m(clip, use_dam=True)
print('eval out shapes:', [tuple(o.shape) for o in outs])


## 10. Phase 1.4 — R0 baseline (two-stage, ~200 epochs)
Auto-resumes from `runs/{RUN_TAG}/fista_last.pth` then `dam_last.pth`. Re-run this
cell after any session timeout. Logs: `runs/{RUN_TAG}/eval_log.csv`.


In [ ]:
%cd {MOCID_DIR}
!python main.py train --tag {RUN_TAG}


## 11. Evaluate

In [ ]:
%cd {MOCID_DIR}
!python main.py eval --ckpt runs/{RUN_TAG}/dam_best.pth  --dam
!python main.py eval --ckpt runs/{RUN_TAG}/fista_best.pth --no-dam


## 12. Record
Copy stage-1 (`fista_best`) and stage-2 (`dam_best`) AP50 / F1 into
[EXPERIMENTS.md](../EXPERIMENTS.md), plus the `python main.py params` numbers,
the GPU type, `git rev-parse --short HEAD`, and any NaN-skip count from the log.
